# Data preprocessing demo

This notebook demonstrates the preprocessing steps:

1. Load reified RDF statements and keep only statements with truth value `1.0`.
2. Extract subject, predicate and object.
3. Fetch Wikipedia article text and outgoing links (optional live step).
4. Clean Wikipedia markup and text.
5. Normalize entity/article names.
6. Hash cleaned text to identify duplicate content.

In [1]:
import hashlib
import html
import re
import requests
from rdflib import Graph, URIRef
from rdflib.namespace import RDF

print("Imports successful")

Imports successful


In [2]:
TRUTH = URIRef("http://swc2017.aksw.org/hasTruthValue")

In [3]:
def load_data_from_graph(graph):
    """Return (subject_uri, predicate_uri, object_uri) for truth value 1.0."""
    data = []

    for stmt in graph.subjects(RDF.type, RDF.Statement):
        truth_value = graph.value(stmt, TRUTH)

        if truth_value is None or str(truth_value) != "1.0":
            continue

        subj = graph.value(stmt, RDF.subject)
        pred = graph.value(stmt, RDF.predicate)
        obj = graph.value(stmt, RDF.object)

        if subj and pred and obj:
            data.append((str(subj), str(pred), str(obj)))

    return data

## Load and filter RDF statements

In [4]:
INPUT_FILE = "../data/input/BPDP_train_reified.nt"

def load_data(input_file):
    graph = Graph()
    graph.parse(input_file, format="nt")
    return load_data_from_graph(graph)

data = load_data(INPUT_FILE)
print("True statements:", len(data))
print("First statement:", data[0] if data else "No matching statements")

True statements: 100
First statement: ('http://dbpedia.org/resource/John_James_Audubon', 'http://www.wikidata.org/prop/direct/P19', 'http://dbpedia.org/resource/Les_Cayes')


## Extract Wikipedia titles from RDF entities

In [5]:
def normalize_filename(filename):
    """Normalize a title similarly to the original script."""
    filename = re.sub(r'[\\\\/:*?"<>|]', '_', filename)
    return filename.replace("_", " ").strip().lower()


subject_uri, predicate_uri, object_uri = data[0]
subject_title = subject_uri.rstrip("/").split("/")[-1]
object_title = object_uri.rstrip("/").split("/")[-1]

subject_normalized = normalize_filename(subject_title)
object_normalized = normalize_filename(object_title)
entity_pair = f"{subject_normalized}_{object_normalized}"

print("Subject title:     ", subject_title)
print("Object title:      ", object_title)
print("Normalized subject:", subject_normalized)
print("Normalized object: ", object_normalized)
print("Entity pair:       ", entity_pair)

Subject title:      John_James_Audubon
Object title:       Les_Cayes
Normalized subject: john james audubon
Normalized object:  les cayes
Entity pair:        john james audubon_les cayes


## 4. Fetch Wikipedia article text and links (optional live step)

This follows the approach in your `fetch_wikipedia_content()` function: call the MediaWiki API, request plain-text extracts and outgoing links, follow redirects, and collect link pages.

Run this cell only when you have internet access. It makes a live request to Wikipedia.

In [6]:
WIKI_HEADERS = {
    "User-Agent": "no-need-to-be-so-big/0.1 (university project)"
}

def fetch_wikipedia_content(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|links",
        "explaintext": 1,
        "redirects": 1,
        "pllimit": "max",
        "titles": title,
    }

    response = requests.get(url, params=params, headers=WIKI_HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    query = payload.get("query", {})

    # Follow a redirect, as in the original script.
    redirects = query.get("redirects", [])
    if redirects:
        return fetch_wikipedia_content(redirects[0]["to"])

    pages = query.get("pages", {})
    page = next(iter(pages.values()), {})

    if "missing" in page:
        return None, []

    article_text = page.get("extract", "")
    links = [item["title"] for item in page.get("links", [])]

    # Retrieve remaining outgoing links if the API indicates another page.
    while "continue" in payload and "plcontinue" in payload["continue"]:
        params["plcontinue"] = payload["continue"]["plcontinue"]
        response = requests.get(url, params=params, headers=WIKI_HEADERS, timeout=30)
        response.raise_for_status()
        payload = response.json()
        for page_data in payload.get("query", {}).get("pages", {}).values():
            links.extend(item["title"] for item in page_data.get("links", []))

    return article_text, links


# Example live request:
article_text, outgoing_links = fetch_wikipedia_content("Albert Einstein")
print(article_text[:1000])
print("Number of outgoing links:", len(outgoing_links))

Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation". He received the 1921 Nobel Prize in Physics for "his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect".
Born as a subject to the Kingdom of Württemberg, part of the German Empire, which he later renounced on 28 January 1896, just 6 weeks before he turned 17, avoiding conscription into military service. The familiy moved to Munich in 1880, where he grew up and attended primary, lower and higher secondary schools. In 1895, he moved to Pavia, Italy, to stay with his family which moved there an year earlier and then further to Swiss canton of Aargau to attend a final year in gymnasium in Aarau. In 1896, 

## Clean Wikipedia text

The `clean_text()` function removes or normalizes Wikipedia formatting before text is sent to the LLM. It:
- decodes HTML entities;
- removes section headings;
- keeps the display text from wiki links;
- removes `<ref>...</ref>` blocks and remaining HTML tags;
- removes simple `{{...}}` templates;
- removes unwanted quote/backslash characters;
- removes parenthetical text containing non-ASCII characters;
- normalizes whitespace while preserving paragraph breaks.

In [8]:
def clean_text(text):
    if not text:
        return ""

    text = html.unescape(text)

    # Remove section headings such as == Early life ==
    text = re.sub(r"(?m)^={2,}.*?={2,}\s*$", "", text)

    # [[Page|display text]] -> display text; [[Page]] -> Page
    text = re.sub(r"\[\[.*?\|(.*?)\]\]", r"\1", text)
    text = re.sub(r"\[\[(.*?)\]\]", r"\1", text)

    # Remove references and HTML tags
    text = re.sub(r"<ref\b[^>]*>.*?</ref>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<[^>]+>", "", text)

    # Remove simple/multiline template blocks (basic approximation)
    text = re.sub(r"\{\{.*?\}\}", "", text, flags=re.DOTALL)

    # Remove backslashes and quote characters, as in the original script
    text = text.replace("\\\\", "").replace("'", "").replace('"', "")

    # Remove parenthetical content if it contains non-ASCII characters
    text = re.sub(r"\([^()]*[^\x00-\x7F][^()]*\)", "", text)

    # Remove empty parentheses
    text = re.sub(r"\(\s*\)", "", text)

    # Normalize spaces/tabs and paragraph spacing
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", "\n\n", text)

    return text.strip()


sample_wikitext = """== Early life ==
Albert Einstein was born in [[Ulm|Ulm, Germany]].
<ref>Example reference text</ref>
He later developed the theory of relativity.
"""

cleaned_sample = clean_text(sample_wikitext)
print("BEFORE CLEANING:\n")
print(sample_wikitext)
print("\nAFTER CLEANING:\n")
print(cleaned_sample)

BEFORE CLEANING:

== Early life ==
Albert Einstein was born in [[Ulm|Ulm, Germany]].
<ref>Example reference text</ref>
He later developed the theory of relativity.


AFTER CLEANING:

Albert Einstein was born in Ulm, Germany.

He later developed the theory of relativity.


## Detect duplicate article content with a hash

`compute_hash()` function calculates an MD5 hash of the cleaned text. Main loop checks whether that hash has already been processed, so identical article text can be skipped.

In [9]:
def compute_hash(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()


hash_1 = compute_hash(cleaned_sample)
hash_2 = compute_hash(clean_text(sample_wikitext))

print("Hash:", hash_1)
print("Same content has same hash:", hash_1 == hash_2)

processed_hashes = {hash_1}
candidate_hash = compute_hash(cleaned_sample)

if candidate_hash in processed_hashes:
    print("Duplicate content: skip processing")
else:
    print("New content: process article")

Hash: ebf6f13a78bade8db48cc4f3d5853ec1
Same content has same hash: True
Duplicate content: skip processing


## End-to-end preprocessing example

This cell brings the main preprocessing functions together using the demonstration RDF statement.

In [11]:
# Use the first true RDF statement from the in-memory example.
sub_uri, pred_uri, obj_uri = data[0]

sub_title = sub_uri.rstrip("/").split("/")[-1]
obj_title = obj_uri.rstrip("/").split("/")[-1]

sub_name = normalize_filename(sub_title)
obj_name = normalize_filename(obj_title)
pair_name = f"{sub_name}_{obj_name}"

# Demonstration article text stands in for the text fetched from Wikipedia.
raw_article = (
    "== Background ==\n"
    "Albert Einstein was born in [[Ulm|Ulm, Germany]].\n"
    "<ref>Example reference</ref>\n"
    "He became a theoretical physicist.\n"
)

cleaned_article = clean_text(raw_article)
content_hash = compute_hash(cleaned_article)

print("Entity pair:", pair_name)
print("\nCleaned article:\n", cleaned_article)
print("\nContent hash:", content_hash)

Entity pair: john james audubon_les cayes

Cleaned article:
 Albert Einstein was born in Ulm, Germany.

He became a theoretical physicist.

Content hash: 6883f804e6b36204c644ebe98c1ce3a7


## Summary:

| Step | Code function | Output |
|---|---|---|
| RDF parsing and truth filtering | `load_data()` | True subject-predicate-object statements |
| Wikipedia extraction | `fetch_wikipedia_content()` | Article text and outgoing links |
| Text cleaning | `clean_text()` | Cleaner plain text |
| Name normalization | `normalize_filename()` | Consistent entity/article names |
| Duplicate detection | `compute_hash()` and processed-hash checks | Detects repeated content |